# Result of source appointment (PMF)

## 春 (MAM)

In [ ]:
# 春季(MAM)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. 自定义配置区 (在这里修改物种排序和源名称)
# ==========================================
def configure_plot_style():
    """配置符合毕业论文要求的全局绘图样式"""
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 1.1 定义你想要的X轴物种排序 (名称必须与CSV文件中的Species列完全一致)
# 注意：这里的物种种类和数目必须和CSV中完全一一对应！
SPECIES_ORDER = [
    'PM2.5', 'NH4+', 'SO42-', 'NO3-', 'Cl-',
    'OC(optical)', 'EC(optical)', 'Pb', 'Zn', 'Cu', 
    'As', 'Fe', 'Mn', 'Ti', 'Ba',
    'Ca', 'K', 'Si', 'Na+', 'Cr', 
    'Ni', 'V', 'Al'
]

# 1.2 定义每个Factor对应的源名称
# 键为CSV中的Factor列名，值为你想要在图上显示的文本。顺序决定了子图从上到下的顺序。
FACTOR_NAMES = {
    'Factor 1': 'Mineral dust',
    'Factor 2': 'Coal combustion',
    'Factor 3': 'Mixed industrial emissions',
    'Factor 4': 'Secondary formation',
    'Factor 5': 'Vehicle emissions',
}

# CSV 文件路径
FILE_PATH = r'D:\Coding\Data\Lanzhou_chemical\PMF\MAM_profiles.csv'

# ==========================================
# 2. 数据读取与解析函数
# ==========================================
def load_pmf_data(filepath):
    """从特殊的PMF CSV格式中提取前两部分(浓度和百分比)数据"""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    conc_data, pct_data = [], []
    state = 0 # 0: 寻找浓度, 1: 读取浓度, 2: 寻找百分比, 3: 读取百分比
    headers = []

    for line in lines:
        line = line.strip()
        if not line: continue
        
        # 判断当前所在的数据块
        if "Factor Profiles (conc. of species)" in line:
            state = 1
            continue
        elif "Factor Profiles (% of species sum)" in line:
            state = 3
            continue
        elif "Factor Profiles (% of total variable)" in line:
            break # 遇到第三部分直接停止读取

        if state == 1 or state == 3:
            if line.startswith(",,Factor"):
                headers = line.split(",")[1:]
                headers[0] = 'Species'
                continue
            
            parts = line.split(",")
            if len(parts) > 2 and parts[1] != 'Species':
                # 提取物种名和数据，跳过缺失值 '*'
                row_data = [parts[1]] + [float(x) if x != '*' else 0.0 for x in parts[2:]]
                if state == 1:
                    conc_data.append(row_data)
                elif state == 3:
                    pct_data.append(row_data)

    df_conc = pd.DataFrame(conc_data, columns=headers)
    df_pct = pd.DataFrame(pct_data, columns=headers)
    
    return df_conc, df_pct

# ==========================================
# 3. 数据校验与绘图执行区
# ==========================================
# 3.1 读取数据
df_conc, df_pct = load_pmf_data(FILE_PATH)

# 将物种列设置为索引
df_conc.set_index('Species', inplace=True)
df_pct.set_index('Species', inplace=True)

# 3.2 校验 SPECIES_ORDER 与 CSV 中的物种是否完全一一对应
csv_species = set(df_conc.index)
order_species = set(SPECIES_ORDER)

if csv_species != order_species:
    print("\n" + "="*50)
    print("数据校验失败：SPECIES_ORDER 与 CSV 中的物种不匹配！")
    
    extra_species = order_species - csv_species
    if extra_species:
        print(f"-> SPECIES_ORDER 中多出了以下物种(CSV中没有): {list(extra_species)}")
        
    missing_species = csv_species - order_species
    if missing_species:
        print(f"-> SPECIES_ORDER 中缺少了以下物种(CSV中存在): {list(missing_species)}")
        
    print("="*50)
    print("请修改代码中的 SPECIES_ORDER 列表，确保两者种类和数目完全一致，然后再运行程序。\n")

# 3.3 按照自定义的 SPECIES_ORDER 重排数据
df_conc = df_conc.loc[SPECIES_ORDER]
df_pct = df_pct.loc[SPECIES_ORDER]

# 3.4 准备X轴标签 (将部分离子和PM2.5格式化为上下标形式)
def format_species_name(name):
    name = name.replace('PM2.5', 'PM$_{2.5}$')
    name = name.replace('NH4+', 'NH$_4^+$')
    name = name.replace('SO42-', 'SO$_4^{2-}$')
    name = name.replace('NO3-', 'NO$_3^-$')
    name = name.replace('Na+', 'Na$^+$')
    name = name.replace('Cl-', 'Cl$^-$')
    name = name.replace('(optical)', '') # 如果不想显示(optical)可以替换掉
    return name

x_labels = [format_species_name(s) for s in df_conc.index]
x_pos = np.arange(len(x_labels))

# 3.5 开始绘图
num_factors = len(FACTOR_NAMES)
fig, axes = plt.subplots(nrows=num_factors, ncols=1, figsize=(8, 2 * num_factors), sharex=True)

if num_factors == 1:
    axes = [axes]

for i, (factor_col, factor_title) in enumerate(FACTOR_NAMES.items()):
    ax1 = axes[i]
    ax2 = ax1.twinx() # 创建共享X轴的第二个Y轴
    
    # 获取当前因子的数据
    conc_values = df_conc[factor_col].values
    pct_values = df_pct[factor_col].values
    
    # 防止log scale时因为0值报错，将0替换为极小值
    conc_values = np.where(conc_values <= 0, 1e-4, conc_values)
    
    # 绘制浓度柱状图 (左轴)
    bar = ax1.bar(x_pos, conc_values, color='darkgray', width=0.6, log=True)
    
    # 绘制百分比散点图 (右轴)
    scatter = ax2.scatter(x_pos, pct_values, color='red', marker='s', s=35, zorder=3)
    
    # 设置Y轴范围和刻度 (根据你截图的样式)
    ax1.set_ylim(1e-3, 1e2)
    ax2.set_ylim(0, 100)
    
    # 在子图内部上方添加源名称 (居中)
    ax1.text(0.5, 0.9, factor_title, transform=ax1.transAxes, 
             ha='center', va='center', fontweight='bold', fontsize=12)
    
    # 设置刻度朝内
    ax1.tick_params(axis='both', direction='in', which='both', labelsize=10)
    ax2.tick_params(axis='y', direction='in', labelsize=10)

# 设置X轴标签 (仅在最底层子图显示)
axes[-1].set_xticks(x_pos)
axes[-1].set_xticklabels(x_labels, rotation=45, fontweight='bold', fontsize=12)

# 设置全局Y轴标签
fig.supylabel(r'Concentration ($\mu$g$\cdot$m$^{-3}$)', fontweight='bold', fontsize=14, x=0.04)
fig.text(0.94, 0.5, '% of Species', va='center', rotation=-90, fontweight='bold', fontsize=14)

# 添加图例 (放置在Figure顶部)
fig.legend([bar, scatter], ['Conc. of Species', '% of Species'], 
           loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.95), 
           fontsize=11, frameon=True, edgecolor='black')

# 调整子图间距
plt.subplots_adjust(hspace=0.1, left=0.15, right=0.88, top=0.90, bottom=0.12)

# 保存和显示图片
plt.savefig(r'D:\Coding\master0_2025\Thesis\PMF_profiles_plot(MAM).png', dpi=500, bbox_inches='tight')
plt.show()

## 夏 (JJA)

In [ ]:
# 夏季
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. 自定义配置区 (在这里修改物种排序和源名称)
# ==========================================
def configure_plot_style():
    """配置符合毕业论文要求的全局绘图样式"""
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 1.1 定义你想要的X轴物种排序 (名称必须与CSV文件中的Species列完全一致)
# 注意：这里的物种种类和数目必须和CSV中完全一一对应！
SPECIES_ORDER = [
    'PM2.5', 'NH4+', 'SO42-', 'NO3-', 'Sn',
    'OC(optical)', 'EC(optical)', 'Pb', 'Zn', 'Cu', 
    'As', 'Fe', 'Mn', 'Ti', 'Ba',
    'Ca', 'K', 'Si', 'Na+', 'Al'
]

# 1.2 定义每个Factor对应的源名称
# 键为CSV中的Factor列名，值为你想要在图上显示的文本。顺序决定了子图从上到下的顺序。
FACTOR_NAMES = {
    'Factor 1': 'Secondary nitrate',
    'Factor 2': 'Vehicle emissions',
    'Factor 3': 'Mixed industrial emissions',
    'Factor 4': 'Mineral dust',
    'Factor 5': 'Secondary sulfate',
}

# CSV 文件路径
FILE_PATH = r'D:\Coding\Data\Lanzhou_chemical\PMF\JJA_profiles.csv'

# ==========================================
# 2. 数据读取与解析函数
# ==========================================
def load_pmf_data(filepath):
    """从特殊的PMF CSV格式中提取前两部分(浓度和百分比)数据"""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    conc_data, pct_data = [], []
    state = 0 # 0: 寻找浓度, 1: 读取浓度, 2: 寻找百分比, 3: 读取百分比
    headers = []

    for line in lines:
        line = line.strip()
        if not line: continue
        
        # 判断当前所在的数据块
        if "Factor Profiles (conc. of species)" in line:
            state = 1
            continue
        elif "Factor Profiles (% of species sum)" in line:
            state = 3
            continue
        elif "Factor Profiles (% of total variable)" in line:
            break # 遇到第三部分直接停止读取

        if state == 1 or state == 3:
            if line.startswith(",,Factor"):
                headers = line.split(",")[1:]
                headers[0] = 'Species'
                continue
            
            parts = line.split(",")
            if len(parts) > 2 and parts[1] != 'Species':
                # 提取物种名和数据，跳过缺失值 '*'
                row_data = [parts[1]] + [float(x) if x != '*' else 0.0 for x in parts[2:]]
                if state == 1:
                    conc_data.append(row_data)
                elif state == 3:
                    pct_data.append(row_data)

    df_conc = pd.DataFrame(conc_data, columns=headers)
    df_pct = pd.DataFrame(pct_data, columns=headers)
    
    return df_conc, df_pct

# ==========================================
# 3. 数据校验与绘图执行区
# ==========================================
# 3.1 读取数据
df_conc, df_pct = load_pmf_data(FILE_PATH)

# 将物种列设置为索引
df_conc.set_index('Species', inplace=True)
df_pct.set_index('Species', inplace=True)

# 3.2 校验 SPECIES_ORDER 与 CSV 中的物种是否完全一一对应
csv_species = set(df_conc.index)
order_species = set(SPECIES_ORDER)

if csv_species != order_species:
    print("\n" + "="*50)
    print("数据校验失败：SPECIES_ORDER 与 CSV 中的物种不匹配！")
    
    extra_species = order_species - csv_species
    if extra_species:
        print(f"-> SPECIES_ORDER 中多出了以下物种(CSV中没有): {list(extra_species)}")
        
    missing_species = csv_species - order_species
    if missing_species:
        print(f"-> SPECIES_ORDER 中缺少了以下物种(CSV中存在): {list(missing_species)}")
        
    print("="*50)
    print("请修改代码中的 SPECIES_ORDER 列表，确保两者种类和数目完全一致，然后再运行程序。\n")

# 3.3 按照自定义的 SPECIES_ORDER 重排数据
df_conc = df_conc.loc[SPECIES_ORDER]
df_pct = df_pct.loc[SPECIES_ORDER]

# 3.4 准备X轴标签 (将部分离子和PM2.5格式化为上下标形式)
def format_species_name(name):
    name = name.replace('PM2.5', 'PM$_{2.5}$')
    name = name.replace('NH4+', 'NH$_4^+$')
    name = name.replace('SO42-', 'SO$_4^{2-}$')
    name = name.replace('NO3-', 'NO$_3^-$')
    name = name.replace('Na+', 'Na$^+$')
    name = name.replace('Cl-', 'Cl$^-$')
    name = name.replace('(optical)', '') # 如果不想显示(optical)可以替换掉
    return name

x_labels = [format_species_name(s) for s in df_conc.index]
x_pos = np.arange(len(x_labels))

# 3.5 开始绘图
num_factors = len(FACTOR_NAMES)
fig, axes = plt.subplots(nrows=num_factors, ncols=1, figsize=(8, 2 * num_factors), sharex=True)

if num_factors == 1:
    axes = [axes]

for i, (factor_col, factor_title) in enumerate(FACTOR_NAMES.items()):
    ax1 = axes[i]
    ax2 = ax1.twinx() # 创建共享X轴的第二个Y轴
    
    # 获取当前因子的数据
    conc_values = df_conc[factor_col].values
    pct_values = df_pct[factor_col].values
    
    # 防止log scale时因为0值报错，将0替换为极小值
    conc_values = np.where(conc_values <= 0, 1e-4, conc_values)
    
    # 绘制浓度柱状图 (左轴)
    bar = ax1.bar(x_pos, conc_values, color='darkgray', width=0.6, log=True)
    
    # 绘制百分比散点图 (右轴)
    scatter = ax2.scatter(x_pos, pct_values, color='red', marker='s', s=35, zorder=3)
    
    # 设置Y轴范围和刻度 (根据你截图的样式)
    ax1.set_ylim(1e-3, 1e2)
    ax2.set_ylim(0, 100)
    
    # 在子图内部上方添加源名称 (居中)
    ax1.text(0.5, 0.9, factor_title, transform=ax1.transAxes, 
             ha='center', va='center', fontweight='bold', fontsize=12)
    
    # 设置刻度朝内
    ax1.tick_params(axis='both', direction='in', which='both', labelsize=10)
    ax2.tick_params(axis='y', direction='in', labelsize=10)

# 设置X轴标签 (仅在最底层子图显示)
axes[-1].set_xticks(x_pos)
axes[-1].set_xticklabels(x_labels, rotation=45, fontweight='bold', fontsize=12)

# 设置全局Y轴标签
fig.supylabel(r'Concentration ($\mu$g$\cdot$m$^{-3}$)', fontweight='bold', fontsize=14, x=0.04)
fig.text(0.94, 0.5, '% of Species', va='center', rotation=-90, fontweight='bold', fontsize=14)

# 添加图例 (放置在Figure顶部)
fig.legend([bar, scatter], ['Conc. of Species', '% of Species'], 
           loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.95), 
           fontsize=11, frameon=True, edgecolor='black')

# 调整子图间距
plt.subplots_adjust(hspace=0.1, left=0.15, right=0.88, top=0.90, bottom=0.12)

# 保存和显示图片
plt.savefig(r'D:\Coding\master0_2025\Thesis\PMF_profiles_plot(JJA).png', dpi=500, bbox_inches='tight')
plt.show()

## 秋 (SON)

In [ ]:
# 秋季(SON)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. 自定义配置区 (在这里修改物种排序和源名称)
# ==========================================
def configure_plot_style():
    """配置符合毕业论文要求的全局绘图样式"""
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 1.1 定义你想要的X轴物种排序 (名称必须与CSV文件中的Species列完全一致)
# 注意：这里的物种种类和数目必须和CSV中完全一一对应！
SPECIES_ORDER = [
    'PM2.5', 'NH4+', 'SO42-', 'NO3-', 'Sn',
    'OC(optical)', 'EC(optical)', 'Pb', 'Zn', 'Cu', 
    'As', 'Fe', 'Mn', 'Ti', 'Ba',
    'Ca', 'K', 'Si', 'Na+', 'Mg2+',
    'Al'
]

# 1.2 定义每个Factor对应的源名称
# 键为CSV中的Factor列名，值为你想要在图上显示的文本。顺序决定了子图从上到下的顺序。
FACTOR_NAMES = {
    'Factor 1': 'Secondary nitrate',
    'Factor 2': 'Mixed industrial emissions',
    'Factor 3': 'Mineral dust',
    'Factor 4': 'Vehicle emissions',
    'Factor 5': 'Power plant',
}

# CSV 文件路径
FILE_PATH = r'D:\Coding\Data\Lanzhou_chemical\PMF\SON_profiles.csv'

# ==========================================
# 2. 数据读取与解析函数
# ==========================================
def load_pmf_data(filepath):
    """从特殊的PMF CSV格式中提取前两部分(浓度和百分比)数据"""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    conc_data, pct_data = [], []
    state = 0 # 0: 寻找浓度, 1: 读取浓度, 2: 寻找百分比, 3: 读取百分比
    headers = []

    for line in lines:
        line = line.strip()
        if not line: continue
        
        # 判断当前所在的数据块
        if "Factor Profiles (conc. of species)" in line:
            state = 1
            continue
        elif "Factor Profiles (% of species sum)" in line:
            state = 3
            continue
        elif "Factor Profiles (% of total variable)" in line:
            break # 遇到第三部分直接停止读取

        if state == 1 or state == 3:
            if line.startswith(",,Factor"):
                headers = line.split(",")[1:]
                headers[0] = 'Species'
                continue
            
            parts = line.split(",")
            if len(parts) > 2 and parts[1] != 'Species':
                # 提取物种名和数据，跳过缺失值 '*'
                row_data = [parts[1]] + [float(x) if x != '*' else 0.0 for x in parts[2:]]
                if state == 1:
                    conc_data.append(row_data)
                elif state == 3:
                    pct_data.append(row_data)

    df_conc = pd.DataFrame(conc_data, columns=headers)
    df_pct = pd.DataFrame(pct_data, columns=headers)
    
    return df_conc, df_pct

# ==========================================
# 3. 数据校验与绘图执行区
# ==========================================
# 3.1 读取数据
df_conc, df_pct = load_pmf_data(FILE_PATH)

# 将物种列设置为索引
df_conc.set_index('Species', inplace=True)
df_pct.set_index('Species', inplace=True)

# 3.2 校验 SPECIES_ORDER 与 CSV 中的物种是否完全一一对应
csv_species = set(df_conc.index)
order_species = set(SPECIES_ORDER)

if csv_species != order_species:
    print("\n" + "="*50)
    print("数据校验失败：SPECIES_ORDER 与 CSV 中的物种不匹配！")
    
    extra_species = order_species - csv_species
    if extra_species:
        print(f"-> SPECIES_ORDER 中多出了以下物种: {list(extra_species)}")
        
    missing_species = csv_species - order_species
    if missing_species:
        print(f"-> SPECIES_ORDER 中缺少了以下物种: {list(missing_species)}")
        
    print("="*50)
    print("请修改代码中的 SPECIES_ORDER 列表，确保两者种类和数目完全一致，然后再运行程序。\n")

# 3.3 按照自定义的 SPECIES_ORDER 重排数据
df_conc = df_conc.loc[SPECIES_ORDER]
df_pct = df_pct.loc[SPECIES_ORDER]

# 3.4 准备X轴标签 (将部分离子和PM2.5格式化为上下标形式)
def format_species_name(name):
    name = name.replace('PM2.5', 'PM$_{2.5}$')
    name = name.replace('NH4+', 'NH$_4^+$')
    name = name.replace('SO42-', 'SO$_4^{2-}$')
    name = name.replace('NO3-', 'NO$_3^-$')
    name = name.replace('Na+', 'Na$^+$')
    name = name.replace('Cl-', 'Cl$^-$')
    name = name.replace('(optical)', '') # 如果不想显示(optical)可以替换掉
    name = name.replace('Mg2+', 'Mg$^{2+}$')
    return name

x_labels = [format_species_name(s) for s in df_conc.index]
x_pos = np.arange(len(x_labels))

# 3.5 开始绘图
num_factors = len(FACTOR_NAMES)
fig, axes = plt.subplots(nrows=num_factors, ncols=1, figsize=(8, 2 * num_factors), sharex=True)

if num_factors == 1:
    axes = [axes]

for i, (factor_col, factor_title) in enumerate(FACTOR_NAMES.items()):
    ax1 = axes[i]
    ax2 = ax1.twinx() # 创建共享X轴的第二个Y轴
    
    # 获取当前因子的数据
    conc_values = df_conc[factor_col].values
    pct_values = df_pct[factor_col].values
    
    # 防止log scale时因为0值报错，将0替换为极小值
    conc_values = np.where(conc_values <= 0, 1e-4, conc_values)
    
    # 绘制浓度柱状图 (左轴)
    bar = ax1.bar(x_pos, conc_values, color='darkgray', width=0.6, log=True)
    
    # 绘制百分比散点图 (右轴)
    scatter = ax2.scatter(x_pos, pct_values, color='red', marker='s', s=35, zorder=3)
    
    # 设置Y轴范围和刻度 (根据你截图的样式)
    ax1.set_ylim(1e-3, 1e2)
    ax2.set_ylim(0, 100)
    
    # 在子图内部上方添加源名称 (居中)
    ax1.text(0.5, 0.9, factor_title, transform=ax1.transAxes, 
             ha='center', va='center', fontweight='bold', fontsize=12)
    
    # 设置刻度朝内
    ax1.tick_params(axis='both', direction='in', which='both', labelsize=10)
    ax2.tick_params(axis='y', direction='in', labelsize=10)

# 设置X轴标签 (仅在最底层子图显示)
axes[-1].set_xticks(x_pos)
axes[-1].set_xticklabels(x_labels, rotation=45, fontweight='bold', fontsize=12)

# 设置全局Y轴标签
fig.supylabel(r'Concentration ($\mu$g$\cdot$m$^{-3}$)', fontweight='bold', fontsize=14, x=0.04)
fig.text(0.94, 0.5, '% of Species', va='center', rotation=-90, fontweight='bold', fontsize=14)

# 添加图例 (放置在Figure顶部)
fig.legend([bar, scatter], ['Conc. of Species', '% of Species'], 
           loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.95), 
           fontsize=11, frameon=True, edgecolor='black')

# 调整子图间距
plt.subplots_adjust(hspace=0.1, left=0.15, right=0.88, top=0.90, bottom=0.12)

# 保存和显示图片
plt.savefig(r'D:\Coding\master0_2025\Thesis\PMF_profiles_plot(SON).png', dpi=500, bbox_inches='tight')
plt.show()

## 冬 (DJF)

In [ ]:
# 冬季(DJF)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. 自定义配置区 (在这里修改物种排序和源名称)
# ==========================================
def configure_plot_style():
    """配置符合毕业论文要求的全局绘图样式"""
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 1.1 定义你想要的X轴物种排序 (名称必须与CSV文件中的Species列完全一致)
# 注意：这里的物种种类和数目必须和CSV中完全一一对应！
SPECIES_ORDER = [
    'PM2.5', 'NH4+', 'SO42-', 'NO3-', 'Cl-',
    'OC(optical)', 'EC(optical)', 'Pb', 'Zn', 'Cu', 
    'As', 'Fe', 'Mn', 'Ti', 'Ba',
    'Ca', 'K', 'Si', 'Mg2+', 'Se',
    'Sb', 'Al'
]

# 1.2 定义每个Factor对应的源名称
# 键为CSV中的Factor列名，值为你想要在图上显示的文本。顺序决定了子图从上到下的顺序。
FACTOR_NAMES = {
    'Factor 1': 'Coal combustion',
    'Factor 2': 'Mineral dust',
    'Factor 3': 'Vehicle emissions',
    'Factor 4': 'Secondary formation',
    'Factor 5': 'Smelting industry',
    'Factor 6': 'Fireworks', 
    'Factor 7': 'Power plant',
}

# CSV 文件路径
FILE_PATH = r'D:\Coding\Data\Lanzhou_chemical\PMF\DJF_profiles.csv'

# ==========================================
# 2. 数据读取与解析函数
# ==========================================
def load_pmf_data(filepath):
    """从特殊的PMF CSV格式中提取前两部分(浓度和百分比)数据"""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    conc_data, pct_data = [], []
    state = 0 # 0: 寻找浓度, 1: 读取浓度, 2: 寻找百分比, 3: 读取百分比
    headers = []

    for line in lines:
        line = line.strip()
        if not line: continue
        
        # 判断当前所在的数据块
        if "Factor Profiles (conc. of species)" in line:
            state = 1
            continue
        elif "Factor Profiles (% of species sum)" in line:
            state = 3
            continue
        elif "Factor Profiles (% of total variable)" in line:
            break # 遇到第三部分直接停止读取

        if state == 1 or state == 3:
            if line.startswith(",,Factor"):
                headers = line.split(",")[1:]
                headers[0] = 'Species'
                continue
            
            parts = line.split(",")
            if len(parts) > 2 and parts[1] != 'Species':
                # 提取物种名和数据，跳过缺失值 '*'
                row_data = [parts[1]] + [float(x) if x != '*' else 0.0 for x in parts[2:]]
                if state == 1:
                    conc_data.append(row_data)
                elif state == 3:
                    pct_data.append(row_data)

    df_conc = pd.DataFrame(conc_data, columns=headers)
    df_pct = pd.DataFrame(pct_data, columns=headers)
    
    return df_conc, df_pct

# ==========================================
# 3. 数据校验与绘图执行区
# ==========================================
# 3.1 读取数据
df_conc, df_pct = load_pmf_data(FILE_PATH)

# 将物种列设置为索引
df_conc.set_index('Species', inplace=True)
df_pct.set_index('Species', inplace=True)

# 3.2 校验 SPECIES_ORDER 与 CSV 中的物种是否完全一一对应
csv_species = set(df_conc.index)
order_species = set(SPECIES_ORDER)

if csv_species != order_species:
    print("\n" + "="*50)
    print("数据校验失败：SPECIES_ORDER 与 CSV 中的物种不匹配！")
    
    extra_species = order_species - csv_species
    if extra_species:
        print(f"-> SPECIES_ORDER 中多出了以下物种: {list(extra_species)}")
        
    missing_species = csv_species - order_species
    if missing_species:
        print(f"-> SPECIES_ORDER 中缺少了以下物种: {list(missing_species)}")
        
    print("="*50)
    print("请修改代码中的 SPECIES_ORDER 列表，确保两者种类和数目完全一致，然后再运行程序。\n")

# 3.3 按照自定义的 SPECIES_ORDER 重排数据
df_conc = df_conc.loc[SPECIES_ORDER]
df_pct = df_pct.loc[SPECIES_ORDER]

# 3.4 准备X轴标签 (将部分离子和PM2.5格式化为上下标形式)
def format_species_name(name):
    name = name.replace('PM2.5', 'PM$_{2.5}$')
    name = name.replace('NH4+', 'NH$_4^+$')
    name = name.replace('SO42-', 'SO$_4^{2-}$')
    name = name.replace('NO3-', 'NO$_3^-$')
    name = name.replace('Na+', 'Na$^+$')
    name = name.replace('Cl-', 'Cl$^-$')
    name = name.replace('(optical)', '') # 如果不想显示(optical)可以替换掉
    name = name.replace('Mg2+', 'Mg$^{2+}$')
    return name

x_labels = [format_species_name(s) for s in df_conc.index]
x_pos = np.arange(len(x_labels))

# 3.5 开始绘图
num_factors = len(FACTOR_NAMES)
fig, axes = plt.subplots(nrows=num_factors, ncols=1, figsize=(8, 2 * num_factors), sharex=True)

if num_factors == 1:
    axes = [axes]

for i, (factor_col, factor_title) in enumerate(FACTOR_NAMES.items()):
    ax1 = axes[i]
    ax2 = ax1.twinx() # 创建共享X轴的第二个Y轴
    
    # 获取当前因子的数据
    conc_values = df_conc[factor_col].values
    pct_values = df_pct[factor_col].values
    # 防止log scale时因为0值报错，将0替换为极小值
    conc_values = np.where(conc_values <= 0, 1e-4, conc_values)
    
    # 绘制浓度柱状图 (左轴)
    bar = ax1.bar(x_pos, conc_values, color='darkgray', width=0.6, log=True)
    
    # 绘制百分比散点图 (右轴)
    scatter = ax2.scatter(x_pos, pct_values, color='red', marker='s', s=35, zorder=3)
    
    # 设置Y轴范围和刻度 (根据你截图的样式)
    ax1.set_ylim(1e-3, 1e2)
    ax2.set_ylim(0, 100)
    
    # 在子图内部上方添加源名称 (居中)
    ax1.text(0.5, 0.9, factor_title, transform=ax1.transAxes, 
             ha='center', va='center', fontweight='bold', fontsize=12)
    
    # 设置刻度朝内
    ax1.tick_params(axis='both', direction='in', which='both', labelsize=10)
    ax2.tick_params(axis='y', direction='in', labelsize=10)

# 设置X轴标签 (仅在最底层子图显示)
axes[-1].set_xticks(x_pos)
axes[-1].set_xticklabels(x_labels, rotation=45, fontweight='bold', fontsize=12)

# 设置全局Y轴标签
fig.supylabel(r'Concentration ($\mu$g$\cdot$m$^{-3}$)', fontweight='bold', fontsize=14, x=0.04)
fig.text(0.94, 0.5, '% of Species', va='center', rotation=-90, fontweight='bold', fontsize=14)

# 添加图例 (放置在Figure顶部)
fig.legend([bar, scatter], ['Conc. of Species', '% of Species'], 
           loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.95), 
           fontsize=11, frameon=True, edgecolor='black')

# 调整子图间距
plt.subplots_adjust(hspace=0.1, left=0.15, right=0.88, top=0.90, bottom=0.12)

# 保存和显示图片
plt.savefig(r'D:\Coding\master0_2025\Thesis\PMF_profiles_plot(DJF).png', dpi=500, bbox_inches='tight')
plt.show()

# % of PM2.5

In [ ]:
# 计算每个季节的PM2.5平均浓度和标准差, 用于源解析结果绘图
import pandas as pd

def calculate_seasonal_stats(df_season):
    df_season["PM2.5"] = pd.to_numeric(df_season["PM2.5"], errors='coerce')
    # 去除缺失值(-999)后计算平均值和标准差
    #print(f"Calculating stats for season with {len(df_season)} data points...")
    df_season = df_season[df_season["PM2.5"] != -999]
    #print(f"After removing missing values, {len(df_season)} data points remain for PM2.5.")
    mean_pm25 = df_season["PM2.5"].mean()
    std_pm25 = df_season["PM2.5"].std()
    print(f"Season - PM2.5 mean: {mean_pm25:.2f} μg/m³, std: {std_pm25:.2f} μg/m³")

df_conc_MAM =pd.read_csv(r"D:\Coding\Data\Lanzhou_chemical\conc_data_MAM.csv")
df_conc_JJA =pd.read_csv(r"D:\Coding\Data\Lanzhou_chemical\conc_data_JJA.csv")
df_conc_SON =pd.read_csv(r"D:\Coding\Data\Lanzhou_chemical\conc_data_SON.csv")
df_conc_DJF =pd.read_csv(r"D:\Coding\Data\Lanzhou_chemical\conc_data_DJF.csv")

calculate_seasonal_stats(df_conc_MAM)
calculate_seasonal_stats(df_conc_JJA)
calculate_seasonal_stats(df_conc_SON)
calculate_seasonal_stats(df_conc_DJF)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ----------------- 配置信息 -----------------
def configure_plot_style():
    """配置符合毕业论文要求的全局绘图样式"""
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()

# 各季节因子命名 (保持你最初的设定)
FACTOR_NAMES = {
    'spring': {
        'Factor 1': 'Mineral dust', 'Factor 2': 'Coal combustion',
        'Factor 3': 'Mixed industrial emissions', 'Factor 4': 'Secondary formation',
        'Factor 5': 'Vehicle emissions'
    },
    'summer': {
        'Factor 1': 'Secondary nitrate', 'Factor 2': 'Vehicle emissions',
        'Factor 3': 'Mixed industrial emissions', 'Factor 4': 'Mineral dust',
        'Factor 5': 'Secondary sulfate'
    },
    'autumn': {
        'Factor 1': 'Secondary nitrate', 'Factor 2': 'Mixed industrial emissions',
        'Factor 3': 'Mineral dust', 'Factor 4': 'Vehicle emissions',
        'Factor 5': 'Power plant'
    },
    'winter': {
        'Factor 1': 'Coal combustion', 'Factor 2': 'Mineral dust',
        'Factor 3': 'Vehicle emissions', 'Factor 4': 'Secondary formation',
        'Factor 5': 'Smelting industry', 'Factor 6': 'Fireworks', 'Factor 7': 'Power plant'
    }
}

# 中心文字 (PM2.5浓度均值和标准差)
CENTER_TEXTS = {
    'winter': r"$\mathbf{65.7 \pm 28.1 \ \mu g \cdot m^{-3}}$",
    'spring': r"$\mathbf{46.7 \pm 44.1 \ \mu g \cdot m^{-3}}$",
    'summer': r"$\mathbf{24.9 \pm 7.6 \ \mu g \cdot m^{-3}}$",
    'autumn': r"$\mathbf{34.3 \pm 19.7 \ \mu g \cdot m^{-3}}$"
}

# 统一各类源的颜色映射 (文献同款色系)
COLOR_MAP = {
    'Mineral dust': '#FFA500',               # 橙色
    'Coal combustion': '#000000',            # 黑色
    'Vehicle emissions': '#808080',          # 灰色
    'Secondary formation': '#0000FF',        # 蓝色 (合并后统一使用此颜色)
    'Smelting industry': '#556B2F',          # 暗橄榄绿
    'Mixed industrial emissions': '#FF4500', # 橙红色
    'Fireworks': '#008000',                  # 绿色
    'Power plant': '#FF00FF'                 # 品红色
}

# 对应的文件名
FILES = {
    'winter': r"D:\Coding\Data\Lanzhou_chemical\PMF\DJF_profiles.csv",
    'spring': r"D:\Coding\Data\Lanzhou_chemical\PMF\MAM_profiles.csv",
    'summer': r"D:\Coding\Data\Lanzhou_chemical\PMF\JJA_profiles.csv",
    'autumn': r"D:\Coding\Data\Lanzhou_chemical\PMF\SON_profiles.csv"
}


# ----------------- 数据读取函数 -----------------
def extract_pm25_percentages(filepath):
    if not os.path.exists(filepath):
        return []
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    in_target_section = False
    for line in lines:
        if "Factor Profiles (% of species sum)" in line:
            in_target_section = True
            continue
        if "Factor Profiles (% of total variable)" in line:
            break
            
        if in_target_section and "PM2.5" in line:
            parts = line.strip().split(',')
            # 过滤掉非数字内容
            values = [float(v) for v in parts[2:] if v.strip() != '']
            return values
    return []


# ----------------- 绘图逻辑 -----------------
fig, axes = plt.subplots(2, 2, figsize=(8, 10))
axes = axes.flatten()

seasons = ['winter', 'spring', 'summer', 'autumn']
titles = ['(a) winter', '(b) spring', '(c) summer', '(d) autumn']

# 用于搜集全局统一的Legend
handles_dict = {}

for i, season in enumerate(seasons):
    ax = axes[i]
    
    # 1. 获取原始数据
    values = extract_pm25_percentages(FILES[season])
    if not values:
        # 兜底测试数据
        values = [100 / len(FACTOR_NAMES[season])] * len(FACTOR_NAMES[season])
        
    factor_dict = FACTOR_NAMES[season]
    original_names = [factor_dict[f"Factor {j+1}"] for j in range(len(values))]
    
    # 2. 合并同类项 (Secondary nitrate & Secondary sulfate -> Secondary formation)
    merged_data = {}
    for name, val in zip(original_names, values):
        if name in ['Secondary nitrate', 'Secondary sulfate']:
            name = 'Secondary formation'
            
        if name in merged_data:
            merged_data[name] += val
        else:
            merged_data[name] = val
            
    # 获取合并后的新名称和值列表
    names = list(merged_data.keys())
    values = list(merged_data.values())
    colors = [COLOR_MAP.get(name, '#CCCCCC') for name in names]
    
    # 3. 绘制环形图
    wedges, texts, autotexts = ax.pie(
        values, 
        colors=colors,
        autopct='%1.0f%%',
        pctdistance=0.8,
        startangle=90,
        wedgeprops=dict(width=0.35, edgecolor='w', linewidth=2)
    )
    
    # 4. 优化百分比文字样式
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(12)
        # 占比小于3%隐藏数字，防止文字重叠
        if float(autotext.get_text().strip('%')) < 3:
            autotext.set_text('')
            
    # 中心文本
    ax.text(0, 0, CENTER_TEXTS[season], ha='center', va='center', fontsize=10)
    # 子图标题
    ax.set_title(titles[i], fontsize=18, fontweight='bold', pad=10)
    
    # 5. 收集 Legend 句柄，使用字典去重
    for w, name in zip(wedges, names):
        if name not in handles_dict:
            handles_dict[name] = w

# ----------------- 图例配置 -----------------
# 提取去重后的句柄和标签
legend_labels = list(handles_dict.keys())
legend_handles = list(handles_dict.values())

fig.legend(
    legend_handles, legend_labels, 
    loc='upper center', 
    bbox_to_anchor=(0.5, 1.05),
    ncol=2, 
    fontsize=14,
    frameon=True,
    edgecolor='lightgray'
)

plt.subplots_adjust(top=0.95, wspace=-0.05, hspace=-0.3)

# 保存图片
plt.savefig(r"D:\Coding\master0_2025\Thesis\PM2.5_Seasonal_Source_Apportionment.png", dpi=600, bbox_inches='tight')
plt.show()